In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv('car_fuel_efficiency_2026.csv')

# 1. 填補缺失值（作業指示：fill missing values with zeros）
df = df.fillna(0)

# 2. 切分資料（作業指定 exact calls）
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# 3. 重置索引
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 4. 取出目標
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

# 5. 從特徵中刪掉目標
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

# 6. DictVectorizer（注意：HW6 指定 sparse=True）
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))

In [4]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

# 找出第一個 split 用的特徵
feature_name = dv.get_feature_names_out()[dt.tree_.feature[0]]
print(feature_name)

model_year


In [5]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = mean_squared_error(y_val, y_pred) ** 0.5
print(f"{rmse:.4f}")

1.8332


In [6]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = mean_squared_error(y_val, y_pred) ** 0.5
    print(f"n_estimators={n}: RMSE={rmse:.3f}")

n_estimators=10: RMSE=1.833
n_estimators=50: RMSE=1.774
n_estimators=100: RMSE=1.770
n_estimators=150: RMSE=1.769


In [7]:
for depth in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred) ** 0.5
        scores.append(rmse)
    print(f"max_depth={depth}: mean RMSE={np.mean(scores):.4f}")

max_depth=10: mean RMSE=1.7544
max_depth=15: mean RMSE=1.7842
max_depth=20: mean RMSE=1.7868
max_depth=25: mean RMSE=1.7863


In [8]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
feature_names = dv.get_feature_names_out()

# 找出重要性最高的「原始特徵」
original_features = [name.split('=')[0] for name in feature_names]
importance_series = pd.Series(importances, index=original_features)
importance_series = importance_series.groupby(level=0).sum()
print(importance_series.sort_values(ascending=False).head())

model_year             0.296107
vehicle_weight         0.203313
fuel_type              0.193555
horsepower             0.078271
engine_displacement    0.061281
dtype: float64


In [9]:
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',  # 回歸任務！
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    
    model = xgb.train(xgb_params, dtrain, num_boost_round=100)
    y_pred = model.predict(dval)
    rmse = mean_squared_error(y_val, y_pred) ** 0.5
    print(f"eta={eta}: RMSE={rmse:.4f}")

eta=0.3: RMSE=1.8265
eta=0.1: RMSE=1.7248
